# arXiv Nhóm 1 — Crawl & Benchmark

One entrypoint for the M1 CPU-only Colab crawl and benchmark. Final mode covers all four assigned ranges (20,000 IDs), every discovered version, and Semantic Scholar references. Sample mode is for validation only.

Drafted with assistance from OpenAI Codex; follows the group's AI disclosure convention.

**Before running:** select CPU/None hardware accelerator and coordinate one active group crawler. HF mode uses the configured private Dataset and HF_TOKEN from Colab Secrets; no Drive output mount is needed. Drive mode remains available as an explicit fallback. Do not run another arXiv crawler in parallel.

Drive-only initialization requires an explicit team-marker flag; HF uses the existing private Dataset. New runs never delete old data. After a runtime reset, select RESUME with the same run ID/settings. Colab can terminate at any time; resume uses the last verified checkpoint. A missing timing summary is disclosed, never silently counted as zero.


HF mode: store HF_TOKEN in Colab Secrets and enable notebook access. The existing Dataset must be private and writable. Default configuration is a 100-ID smoke, not the final benchmark. Crawl files stay raw, sharded only on HF; resume restores the flat local corpus layout. Every independent run needs a distinct RUN_ID. Never resume the same run concurrently. SHIFT markers are no longer used. The rate limiter is local to each runtime, so parallel arXiv crawls are not yet coordinated; run only one crawler at a time until a shared limiter is available. The latest revision still needs a live smoke and resume test with the chosen backend.


In [ ]:
# Edit this cell. Use a NEW run ID for each new experiment.
MODE = "sample"  # sample, partition (partial assigned range), or final
RUN_ID = "m1-smoke-toggle-01"
USE_HUGGINGFACE = True  # @param {type:"boolean"}
if type(USE_HUGGINGFACE) is not bool:
    raise TypeError("USE_HUGGINGFACE must be True or False")
STORAGE_BACKEND = "huggingface" if USE_HUGGINGFACE else "drive"
HF_REPO_ID = "PancakeDang/arxiv-nhom1-introds"
RANGES = ("dang",)
OFFSET = 0
COUNT = 100  # final: MODE="final", all four ranges, COUNT=None
CHUNK_SIZE = 500  # COUNT=100 still makes one 100-ID smoke chunk
WITH_REFERENCES = True
SNAPSHOT_PATH = "/content/arxiv-assigned-20000.jsonl"
SNAPSHOT_DRIVE_PATH = "/content/drive/MyDrive/arxiv-inputs/arxiv-assigned-20000.jsonl"
SNAPSHOT_SHA256 = "033d7bded327d3aa1595f5502ebfc44bcfe353a89a2011b297742d287ba4de15"
CHECKPOINT_S = 300
RESUME = False  # True in a new runtime after a saved run/timeout
DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"
TEAM_ID = "arxiv-nhom1"
INITIALIZE_TEAM_FOLDER = False  # True once, after checking the intended Drive folder
CODE_REF = "SET_PUBLISHED_REVISION"  # replace with the published implementation commit SHA
REPO = "https://github.com/DankDaPancake/CodeRepo.git"

In [ ]:
# Prevent a new checkout from being mislabeled while Python keeps old src modules.
import sys

if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError(
        "Restart runtime before setup; cached pipeline modules may belong to another revision."
    )

## Copy the filtered snapshot from Drive (before HF authorization)

Upload `arxiv-assigned-20000.jsonl` beside the full snapshot in your Drive input folder. Default: `MyDrive/arxiv-inputs/arxiv-assigned-20000.jsonl`; edit SNAPSHOT_DRIVE_PATH if your folder differs. This 39.99 MB file contains all 20,000 assigned IDs and 26,491 versions, retaining all original fields.

Run this cell to mount Drive for input only and copy the filtered file to /content. The SHA-256 is checked before publishing the local file; a matching existing local file is reused. The Drive original is never modified or deleted. HF_TOKEN is read in the next section only when HF output is selected; crawl outputs/checkpoints go to the chosen backend. No Kaggle download or Kaggle token is needed.

Use a fresh runtime so the full snapshot is absent from local disk. Copy/hash is setup time outside the benchmark; the filtered input is counted in benchmark disk measurements. The filtered file derives from the earlier local snapshot, not the newer HF smoke input: start a NEW RUN_ID when switching to it. For resume keep the same filtered bytes, path and run configuration.


In [ ]:
import hashlib
import json
import shutil
import tempfile
from pathlib import Path


def verify_snapshot(path):
    if not path.is_file():
        raise FileNotFoundError(f"Upload the filtered snapshot to: {path}")
    with path.open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    if actual != SNAPSHOT_SHA256:
        raise ValueError(f"Snapshot SHA-256 mismatch: {path}; check the input file")


if SNAPSHOT_PATH:
    snapshot_target = Path(SNAPSHOT_PATH)
    if snapshot_target.exists():
        verify_snapshot(snapshot_target)
        print("Reusing verified local snapshot:", snapshot_target)
    else:
        from google.colab import drive

        drive.mount("/content/drive")
        snapshot_source = Path(SNAPSHOT_DRIVE_PATH)
        if not snapshot_source.is_file():
            raise FileNotFoundError(
                f"Upload arxiv-assigned-20000.jsonl to {snapshot_source}; "
                "or edit SNAPSHOT_DRIVE_PATH to match its location."
            )
        snapshot_target.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(
            prefix="arxiv-input-", dir=snapshot_target.parent
        ) as temporary:
            copied = Path(temporary) / "snapshot.jsonl"
            shutil.copyfile(snapshot_source, copied)
            verify_snapshot(copied)
            copied.replace(snapshot_target)
        print("Copied and verified filtered snapshot:", snapshot_target)
    print("Snapshot bytes:", snapshot_target.stat().st_size)
else:
    print("Snapshot disabled; discovery will use the network fallback.")

## Authorize selected output storage

HF uses HF_TOKEN from Colab Secrets and a private Dataset; Drive uses the selected account and existing team folder. Drive may already be mounted for the input snapshot.


In [ ]:
import os
import subprocess
import sys
import uuid
from pathlib import Path

from google.colab import userdata

if STORAGE_BACKEND == "huggingface":
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()
    except Exception as exc:
        raise RuntimeError("Add HF_TOKEN in Colab Secrets and enable notebook access") from exc
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.34,<2"], check=True
    )
    os.environ["HF_XET_CACHE"] = "/content/m1-hf-xet-cache"
    from huggingface_hub import HfApi, hf_hub_download

    hf_api = HfApi(token=os.environ["HF_TOKEN"])
    hf_info = hf_api.repo_info(HF_REPO_ID, repo_type="dataset")
    if not hf_info.private:
        raise ValueError("Expected the configured private Dataset")
    print("Private Dataset accessible:", HF_REPO_ID)
    print("Write permission will be checked by the run reservation before crawling")
    DRIVE = Path("/content/m1-hf-staging") / uuid.uuid4().hex
else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path(DRIVE_DIR)
    if not DRIVE.is_dir():
        raise FileNotFoundError("Create/check the intended team Drive folder before running")

## Resolve saved revision

For resume, fetch the saved manifest from the chosen storage. Configuration and snapshot identity must match; keep the same backend and run ID.


In [ ]:
revision = CODE_REF
if RESUME:
    if STORAGE_BACKEND == "huggingface":
        manifest_path = hf_hub_download(
            HF_REPO_ID,
            f"runs/{RUN_ID}/MANIFEST.json",
            repo_type="dataset",
            token=os.environ["HF_TOKEN"],
        )
    else:
        manifest_path = DRIVE / "runs" / RUN_ID / "MANIFEST.json"
    saved = json.loads(Path(manifest_path).read_text())
    revision = saved["code_revision"]

## Check out the implementation

The crawler is pinned to the immutable revision shown below. Restart the runtime before changing revisions.


In [ ]:
if revision == "SET_PUBLISHED_REVISION":
    raise ValueError("Publish the updated implementation and set CODE_REF to its commit SHA")

repo_dir = Path("/content/CodeRepo")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", revision], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Implementation revision:", REVISION)

## Install dependencies

Package installation is setup and is outside the benchmark session.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True
)
if not Path("/usr/bin/time").is_file():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "time"], check=True)

## Optional API credentials and local workspace

ARXIV_CONTACT_EMAIL and S2_API_KEY are read from Colab Secrets if present. No values are printed.


In [ ]:
for name in ("ARXIV_CONTACT_EMAIL", "S2_API_KEY"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value.strip()
    print(name, "configured" if value else "not configured")
LOCAL_ROOT = Path("/content/m1-local") / uuid.uuid4().hex
if SNAPSHOT_PATH and not Path(SNAPSHOT_PATH).is_file():
    raise FileNotFoundError("Copy the filtered snapshot to SNAPSHOT_PATH before benchmarking")

## Preflight and exact scope

Check the displayed ID count, code revision, CPU runtime and storage destination before the long-running cell. A 100-ID Dang sample spans 2604.20535–2604.20634. Final mode rejects partial ranges or references disabled.


In [ ]:
from dataclasses import asdict

import psutil

from src.benchmark.checkpoints import prepare_drive
from src.benchmark.runner import BenchmarkConfig, plan_chunks, run_benchmark

if STORAGE_BACKEND == "drive":
    prepare_drive(DRIVE, TEAM_ID, initialize=INITIALIZE_TEAM_FOLDER)
cfg = BenchmarkConfig(
    run_id=RUN_ID,
    mode=MODE,
    ranges=tuple(RANGES),
    offset=OFFSET,
    count=COUNT,
    chunk_size=CHUNK_SIZE,
    references=WITH_REFERENCES,
    snapshot=SNAPSHOT_PATH,
    storage_backend=STORAGE_BACKEND,
    hf_repo_id=HF_REPO_ID if STORAGE_BACKEND == "huggingface" else None,
    checkpoint_s=CHECKPOINT_S,
)
chunks = plan_chunks(cfg)

## Review scope and runtime

Confirm the CPU runtime, selected IDs and destination before starting. Use a fresh RUN_ID for a new experiment.


In [ ]:
print(asdict(cfg))
print("Selected IDs:", sum(len(c.ids) for c in chunks), "chunks:", len(chunks))
print("CPU count:", psutil.cpu_count(), "RAM MB:", psutil.virtual_memory().total / 1e6)
print("Storage:", STORAGE_BACKEND, HF_REPO_ID if STORAGE_BACKEND == "huggingface" else DRIVE)
subprocess.run(["lscpu"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)

## Crawl, checkpoint and benchmark

Each chunk uses the existing crawler under the process-tree profiler and `/usr/bin/time -v`. Periodic checkpoints run inside the measured crawler. Post-chunk sync is reported separately. Retry and backoff remain in wall time; do not add them again. Setup/install and offline gaps between sessions are excluded from pipeline time.

HF commits target 500 files or 32 MiB of input per batch (oversized individual files go alone). This exceeds HF's usual 50-100-file recommendation and needs live smoke validation against the HTTP validation timeout. Raw files are retained. Unchanged control files and a redundant final report checkpoint are skipped.

HF limits commit attempts through a shared local budget and retries 429 after a logged server cooldown (one-hour fallback). Do not start another runtime to bypass a quota wait.

Checkpoint target: five minutes at completed-version boundaries. A large version or failed Drive upload can exceed that interval. Only completed storage checkpoints survive a destroyed runtime. A file being downloaded may need to restart after runtime loss; live HTTP Range recovery is not a cross-runtime promise.

Stop only after a verified checkpoint when possible. An interrupted version may need to be downloaded again.

With SNAPSHOT_PATH set, paper discovery uses Kaggle first. Only missing/invalid records and historical version titles/authors require arXiv metadata requests. The snapshot fixes version coverage as of its release; do not claim it reflects later updates. A verified index for the selected IDs is built once per run and restored with checkpoints. Setup hashing, index preparation, and crawl timing are reported separately; index preparation is included in session resource measurements. Copy the filtered snapshot from Drive to /content in the preparation cell; do not download the full Kaggle snapshot for this run.


In [ ]:
result = run_benchmark(
    cfg,
    drive_root=DRIVE,
    local_root=LOCAL_ROOT,
    revision=REVISION,
    team_id=TEAM_ID,
    resume=RESUME,
)
print(json.dumps(result, indent=2))
print("Local corpus for M2:", LOCAL_ROOT / RUN_ID / "data/corpus")
print(
    "Durable raw corpus:",
    f"https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/runs/{RUN_ID}/data/corpus"
    if STORAGE_BACKEND == "huggingface"
    else DRIVE / "runs" / RUN_ID / "data/corpus",
)

## Inspect completion and measurements

Corpus coverage and timing completeness are separate. Session RSS includes the notebook/runner baseline and descendants; memory is sampled, so peaks can be missed. Session sampler wall includes crawl, retries and final corpus sync, but excludes initial manifest hash, input setup and final evidence publication. Discovery includes index preparation; do not add overlapping checkpoint or reference times to wall. Mean wall per required ID differs from per-paper lifecycle latency. Preserve logs and measurements even if finalization fails.


In [ ]:
import pandas as pd
from IPython.display import display

local = LOCAL_ROOT / RUN_ID
for name in ("coverage.json", "timing_audit.json"):
    p = local / "work" / name
    if p.exists():
        print(name, p.read_text())
audit = local / "book_csv/measurement_audit.json"
if audit.exists():
    print("Measurement audit:", audit.read_text())
errors = local / "work/cleanup_errors.json"
if errors.exists():
    print("Cleanup diagnostics:", errors.read_text())
for name in ("benchmark_summary.csv", "corpus_summary.csv"):
    p = local / "book_csv" / name
    if p.exists():
        print(name)
        display(pd.read_csv(p))
p = local / "book_csv/paper_attempt_timings.csv"
if p.exists():
    print("Per-paper attempts (first 10; all rows in ZIP)")
    display(pd.read_csv(p).head(10))

## Export and download benchmark evidence

The evidence ZIP contains reports and logs. Raw source files remain in storage.


In [ ]:
from google.colab import files

from src.benchmark.runner import export_evidence, export_rescue

local = LOCAL_ROOT / RUN_ID
try:
    archive = export_evidence(cfg, local)
except Exception as exc:
    print("Statistics export failed; creating raw local rescue:", exc)
    archive = export_rescue(local)
print("Evidence:", archive)
files.download(str(archive))

## Rescue local data after an error

If benchmark/finalization fails, run this cell before resetting the runtime. It packages local data, logs and measurements without contacting the output storage.


In [ ]:
from google.colab import files as rescue_files

from src.benchmark.runner import export_rescue as make_rescue

archive = make_rescue(LOCAL_ROOT / RUN_ID)
print("Raw local rescue:", archive)
rescue_files.download(str(archive))

## Resume in a fresh runtime

Keep RUN_ID, storage backend, chunk size, snapshot bytes/path and settings unchanged; set RESUME=True. Setup loads the saved implementation revision. A run created with an older revision resumes that revision; it does not acquire newer fixes. Confirm the prior runtime has stopped before resuming the same RUN_ID. SHIFT is no longer used.

Drive: the index describes the last committed generation. Changed files retain SHA-256 backups so restore can recover that generation if the plain mirror was updated before index publication. Existing runs that were already corrupted cannot be repaired automatically. The raw Drive mirror may be partially updated after an interruption; restore and verify before M2 consumption.

HF: restore uses the pinned completed LATEST revision, not an in-progress main tree. Empty version directories are reconstructed. Remote corpus paths are sharded; local paths after restore are flat. Session gaps and missing timing tails remain disclosed. Wait for final report publication before handing over.
